# Experimento: FT-Transformer, fase 3 (critério de parada e paciência)

**O que a fase 2 mostrou.** O dropout acabou com o sobreajuste: com ele, a entropia cruzada na separação de parada deixou de subir, e a rede passou a treinar de 15 a 23 épocas em vez de 6. Mesmo assim, o KS não saiu do ruído. Até aqui a parada olhava **só a entropia cruzada (CE)**, e as curvas mostraram duas coisas:

- a CE da parada oscila uns ±0,05 de uma época para outra, e com paciência 16 algumas dobras paravam na época 1;
- com dropout pesado, o maior KS da parada (0,545, na época 78) caiu longe da época de menor CE (96), onde o KS era 0,527.

## Pergunta

**Parar pelo KS, e com mais paciência, sobe o KS do placar?**

| Eixo | Valores |
|---|---|
| Critério de parada | perda (CE) ou KS, os dois medidos nos 20% internos da dobra |
| Paciência | 16 (a do artigo), 32 ou 64 |
| Ponto de partida | trial 0 e `r_drop_medio` (dropout 0,3 / 0,3 / 0,1, que treinou mais sem perder estabilidade) |

São 2 × 3 × 2 = 12 células. As duas com perda e paciência 16 são as próprias bases, que vêm da fase 2 sem treinar de novo.

**Sem vazamento.** O KS que decide a parada é o dos 20% internos. A rede retreina no treino inteiro da dobra pelo número de épocas encontrado, e a dobra externa só mede. Na fase 3 do MLP, a mesma dobra decidia a parada e media o KS, o que inflou o resultado em +0,009. Aqui isso não acontece.

Cada versão leva dois vereditos pareados: contra o trial 0 e contra a **própria base**. É o segundo que responde se a mudança de parada ajudou.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_fase3.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-fase3-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE2_RUN = "ft-fase2-20261115-0024"   # execução da fase 2 na branch de resultados (ou uma pasta local): traz as fases 1 e 2 já medidas
EPOCAS_CURVA = 100              # épocas das curvas (sem parada); as que a fase 2 já fez são reaproveitadas
CURVAS = ("trial0", "r_drop_medio", "r_drop_pesado", "r_combo_pesado")   # configurações nas curvas
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-fase3-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    EPOCAS_CURVA = 3
    print("MODO_TESTE: 3 épocas e duas células da grade. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2, fase3 as F3

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase=3, dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas das fases 1 e 2 (com o trial 0 e o r_drop_medio) e as curvas da fase 2 entram aqui sem treinar de novo
import tempfile
if FASE2_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE2_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE2_RUN) else \
            fluxo.baixar_github(_tmp, FASE2_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
        if ok and os.path.isfile(os.path.join(_tmp, "curvas_regularizacao.json")):
            shutil.copy(os.path.join(_tmp, "curvas_regularizacao.json"), os.path.join(PASTA, "curvas.json"))
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fases 1 e 2: {len(anteriores)} linhas trazidas de {FASE2_RUN}" if anteriores else "Fases 1 e 2: nada trazido; as bases serão medidas aqui")


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F1.tabela(linhas, campos).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")

# 1. Onde cada regra pararia?

As curvas treinam na parte de ajuste da dobra 0 por 100 épocas, **sem parar**. Elas anotam por época a CE no treino, a CE na parada e o KS na parada. As que a fase 2 já fez vêm prontas, e só a do `r_drop_medio` é treinada aqui.

Em cima dessas curvas, a parada é **simulada**: para cada critério e paciência, em que época ela pararia e qual era o KS da parada nesse ponto. A figura mostra o KS por época e marca os pontos escolhidos pela perda (azul) e pelo KS (laranja).

Esse KS é o da mesma separação que escolhe a época, então é otimista para a parada pelo KS. Ele serve para ver o mecanismo. Quem decide é o placar, na seção 2.

As curvas também vão para o W&B, em *curvas/…*, uma com o KS e outra com a CE por configuração.

In [ ]:
CONFIGS = {"trial0": F1.TRIAL0, **F2.REGULARIZACAO}
arq_curvas = os.path.join(PASTA, "curvas.json")
cvs = json.load(open(arq_curvas)) if os.path.isfile(arq_curvas) else {}
cvs = {r: v for r, v in cvs.items() if len(v["ks_parada"]) >= EPOCAS_CURVA}      # curva mais curta (MODO_TESTE) não serve
t0 = time.time()
for r in CURVAS:
    if r not in cvs and continuar():
        cvs[r] = F2.curvas(CONFIGS[r], k=0, epocas=EPOCAS_CURVA, dispositivo=DISPOSITIVOS[0])
        json.dump(cvs, open(arq_curvas, "w"))
print(f"{(time.time() - t0) / 60:.1f} min")
mostrar_cv = {r: cvs[r] for r in CURVAS if r in cvs}
F3.figura_ks(mostrar_cv, os.path.join(PASTA, "curvas_ks.png")); plt.show()
F2.figura_curvas(mostrar_cv, os.path.join(PASTA, "curvas_ce.png")); plt.show()
sim = pd.DataFrame([dict(config=r, **x) for r, v in mostrar_cv.items() for x in F3.paradas_na_curva(v)])
display(sim.pivot(index=["config", "parada"], columns="paciencia", values=["epoca", "ks_parada"]).round(4))
F3.logar_curvas(wandb_run, mostrar_cv)
_ = salvar()

# 2. Parar pelo KS ou com mais paciência sobe o KS do placar?

Cada célula passa pela régua das fases 1 e 2 (15 dobras, parada nos 20% internos, veredito pareado). Vale olhar três colunas:

- **`vs_base`:** o veredito contra a mesma configuração parando pela perda com paciência 16. É ele que diz se a regra de parada ajudou;
- **`epocas`:** a época mediana em que a parada parou;
- **`minutos`:** quanto custou. Com paciência 64, cada dobra treina pelo menos 64 épocas a mais.

In [ ]:
T0 = ("ft_transformer", "trial0")
GRADE = F3.grade_parada()
if MODO_TESTE:
    GRADE = {r: g for r, g in GRADE.items() if r in ("trial0", "r_drop_medio", "s_trial0_ks_p16", "s_r_drop_medio_ks_p32")}
itens = [(r, c, None if r == "trial0" else {"vs_trial0": T0, "vs_base": ("ft_transformer", b)} if r != b else {"vs_trial0": T0})
         for r, (b, _, _, c) in sorted(GRADE.items(), key=lambda kv: kv[0] not in F3.BASES)]       # as bases entram antes
t0 = time.time()
linhas = medir(itens)
print(f"{(time.time() - t0) / 60:.1f} min")
tg = F1.tabela(linhas, ("vs_trial0", "vs_base"))
meta = pd.DataFrame([dict(rotulo=r, base=b, parada=pa, paciencia=pc) for r, (b, pa, pc, _) in GRADE.items()])
tg = meta.merge(tg, on="rotulo").drop(columns=["blocos", "d_token", "over"])
for col, nome in (("ks_kfold", "KS do placar"), ("epocas", "época mediana da parada"), ("minutos", "minutos na GPU")):
    print(nome)
    display(tg.pivot(index=["base", "parada"], columns="paciencia", values=col).round(4))
print("Veredito contra a própria base (perda, paciência 16)")
display(tg.pivot(index=["base", "parada"], columns="paciencia", values="vs_base"))
display(tg.round(4))

# 3. Placar das fases 1 a 3

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
tab = F1.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_ft_transformer_fase3.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do trial 0: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 3

- **Nas curvas, a parada pelo KS escolhe épocas diferentes da parada pela perda?** _…_
- **Alguma célula ganhou da própria base no placar?** _…_
- **Mais paciência ajudou mais na perda ou no KS?** _…_
- **O custo em minutos compensa?** _…_
- **Critério e paciência que seguem para as próximas fases:** _…_